# Quickstart — Browserless × LlamaIndex (Python)

Connect to the Browserless MCP server, list its tools, and run a `FunctionAgent` that scrapes the web. Browserless ships 10 tools over a single streamable-HTTP MCP endpoint: 8 stateless (smartscraper, search, map, crawl, export, performance, function, download) plus a multi-turn browser agent (`browserless_agent` + `browserless_skill`).

Get a token at [account.browserless.io](https://account.browserless.io).

In [ ]:
%pip install -q llama-index llama-index-tools-mcp llama-index-llms-anthropic

In [ ]:
import os
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

client = BasicMCPClient(
    "https://mcp.browserless.io/mcp",
    headers={"Authorization": f"Bearer {os.environ['BROWSERLESS_TOKEN']}"},
    timeout=120,
)

spec = McpToolSpec(client=client)
tools = await spec.to_tool_list_async()
print(f"Loaded {len(tools)} tools:")
for t in tools:
    print(f"  - {t.metadata.name}")

## Direct tool invocation

Each MCP tool wraps as a LlamaIndex `FunctionTool`. Call `browserless_smartscraper` directly to fetch a deterministic URL as markdown.

In [ ]:
smartscraper = next(t for t in tools if t.metadata.name == "browserless_smartscraper")
result = await smartscraper.acall(url="https://example.com", formats=["markdown"])
print(str(result)[:500])

## FunctionAgent (stateless tools)

Hand the stateless tools to a LlamaIndex `FunctionAgent` powered by Claude. Every step is a single self-contained call, so this works regardless of session-id behavior.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.anthropic import Anthropic

stateless_tools = [
    t for t in tools
    if t.metadata.name not in ("browserless_agent", "browserless_skill")
]

agent = FunctionAgent(
    name="BrowserlessAgent",
    description="Browses and scrapes the web via the Browserless MCP server.",
    llm=Anthropic(model="claude-sonnet-4-6"),
    tools=stateless_tools,
    system_prompt="You are a helpful research assistant that uses the Browserless tools to scrape and summarize the web.",
)

response = await agent.run(
    "Scrape https://news.ycombinator.com and list the top 5 headlines as a markdown bullet list."
)
print(response)